# Jalon 1 - Cluster, RDD et Spark SQL

**Public :** binôme du module Écosystème Big Data 2. **Prérequis :** Docker Compose démarré, notions de Python et SQL.

Ce notebook exécute les scripts du dépôt et affiche les résultats essentiels. Ouvrir les fichiers `src/part1_cluster.py`, `src/part2_rdd.py` et `src/part3_sql.py` pour étudier les transformations.

1. Préparer les données et le stockage ; 2. vérifier le cluster ; 3. analyser en RDD ; 4. comparer DataFrames/SQL ; 5. contrôler les résultats.

Les données sont un **remplacement synthétique autorisé**, pas les fichiers officiels de l'enseignant. Les analyses comptent les événements DELIVERED, doublons inclus ; le dédoublonnage streaming appartient au jalon 2.

In [1]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, Markdown
ROOT = Path('/workspace')
def read_result(name):
    return json.loads((ROOT / 'results' / name).read_text(encoding='utf-8'))
RUN_PIPELINE = False  # True pour relancer les calculs ; False pour consulter les preuves enregistrees.

def run_script(name, *args):
    if not RUN_PIPELINE:
        print(f"{name} : resultats enregistres (activer RUN_PIPELINE pour recalculer)")
        return
    completed = subprocess.run([sys.executable, str(ROOT / 'src' / name), *args], cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (ROOT / 'results' / (name + '.log')).write_text(completed.stdout, encoding='utf-8')
    if completed.returncode:
        print(completed.stdout[-12000:])
        raise RuntimeError(f'{name} a échoué')
    print(f'{name} : OK')
def table(rows):
    keys = list(rows[0])
    display(Markdown('| ' + ' | '.join(keys) + ' |\n| ' + ' | '.join(['---'] * len(keys)) + ' |\n' + '\n'.join('| ' + ' | '.join(str(r[k]) for k in keys) + ' |' for r in rows)))

## Données et ressources
Graine fixe 2026, 50 000 événements, 200 restaurants, 2 % de doublons exacts, environ 5 % de retards jusqu'à 15 minutes. Le bootstrap crée le bucket `lake` et le topic `orders` (3 partitions), puis copie les fichiers dans Bronze.

In [2]:
run_script('producer.py', '--mode', 'file', '--n', '50000')
run_script('bootstrap.py')
read_result('bootstrap.json')

producer.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)
bootstrap.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)


{'bucket': 'lake',
 'topic': 'orders',
 'partitions': 3,
 'objects': [{'key': 'bronze/history/generation_manifest.json', 'bytes': 1487},
  {'key': 'bronze/history/orders_history.jsonl', 'bytes': 24936153},
  {'key': 'bronze/reference/restaurants.json', 'bytes': 30482},
  {'key': 'silver/orders_history/_SUCCESS', 'bytes': 0},
  {'key': 'silver/orders_history/city=Casablanca/part-00000-3304acbc-21cc-4a2d-baac-e0c4dc05ac61.c000.snappy.parquet',
   'bytes': 85923},
  {'key': 'silver/orders_history/city=Casablanca/part-00001-3304acbc-21cc-4a2d-baac-e0c4dc05ac61.c000.snappy.parquet',
   'bytes': 84652},
  {'key': 'silver/orders_history/city=Casablanca/part-00002-3304acbc-21cc-4a2d-baac-e0c4dc05ac61.c000.snappy.parquet',
   'bytes': 76116},
  {'key': 'silver/orders_history/city=Fes/part-00000-3304acbc-21cc-4a2d-baac-e0c4dc05ac61.c000.snappy.parquet',
   'bytes': 85858},
  {'key': 'silver/orders_history/city=Fes/part-00001-3304acbc-21cc-4a2d-baac-e0c4dc05ac61.c000.snappy.parquet',
   'bytes': 

## Partie 1 - Cluster
L'action de somme s'exécute sur le cluster. Deux Workers doivent être actifs. `defaultParallelism` est un nombre de partitions par défaut : on affiche séparément les cœurs déclarés par les Workers.

In [3]:
run_script('part1_cluster.py')
read_result('part1_cluster.json')

part1_cluster.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)


{'spark_version': '3.5.6',
 'master': 'spark://spark-master:7077',
 'app_name': 'Jalon1-Partie1-Cluster',
 'application_id': 'app-20261114105747-0000',
 'default_parallelism': 4,
 'python': '3.8.10',
 'alive_workers': 2,
 'worker_cores': 4,
 'test_sum': 499500,
 'note': 'defaultParallelism is a default number of partitions, not a CPU inventory.'}

## Partie 2 - RDD
`textFile`, `map`, `filter` et `reduceByKey` construisent un calcul différé. `collect`, `count` et `takeOrdered` déclenchent son exécution. Le panier moyen utilise un couple (somme, nombre), agrégé en une seule réduction.

In [4]:
run_script('part2_rdd.py')
p2 = read_result('part2_rdd.json')
table(p2['cities'])
table(p2['top5_restaurants'])
print((ROOT / 'results/part2_lineage.txt').read_text())

part2_rdd.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)


| city | revenue_cents | revenue_usd | delivered_events | average_usd |
| --- | --- | --- | --- | --- |
| Market-1 | 5275836 | 52758.36 | 1667 | 31.64868626274745 |
| Market-2 | 31459475 | 314594.75 | 12284 | 25.61012292412895 |
| Market-3 | 722468 | 7224.68 | 235 | 30.74331914893617 |
| Market-4 | 29056648 | 290566.48 | 9780 | 29.710274028629854 |
| Market-5 | 924622 | 9246.22 | 389 | 23.769203084832906 |
| Market-6 | 1456181 | 14561.81 | 649 | 22.437303543913714 |

| restaurant_id | delivered_events |
| --- | --- |
| DD-2-6865 | 387 |
| DD-2-314 | 381 |
| DD-2-1311 | 359 |
| DD-2-1686 | 331 |
| DD-2-6074 | 319 |

(4) PythonRDD[11] at collect at src/part2_rdd.py:18 []
 |  MapPartitionsRDD[10] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[9] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[8] at reduceByKey at src/part2_rdd.py:17 []
    |  PythonRDD[7] at reduceByKey at src/part2_rdd.py:17 []
    |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
    |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []


## Partie 3 - DataFrames et SQL
Schémas explicites, jointure broadcast du référentiel de 200 lignes, comparaison UDF/native, export Parquet partitionné par `city` et relecture. Les mesures comprennent une chauffe puis trois répétitions avec ordre alterné. Une action consomme la colonne calculée pour empêcher Spark d'éliminer l'UDF du test.

In [5]:
run_script('part3_sql.py')
p3 = read_result('part3_sql.json')
table([{'API': name, 'médiane (s)': round(value['median_seconds'], 4)} for name, value in p3['benchmarks'].items()])
table(p3['revenue_by_category'])
print('Lignes Silver relues :', p3['silver_rows'])
print('Stratégie :', p3['join_strategy'])

part3_sql.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)


| API | médiane (s) |
| --- | --- |
| RDD | 0.9216 |
| DataFrame | 0.5746 |
| SQL | 0.7587 |
| UDF | 0.8683 |
| Native | 0.5917 |

| city | category | revenue_usd |
| --- | --- | --- |
| Market-1 | american | 13427.18 |
| Market-1 | barbecue | 2247.00 |
| Market-1 | breakfast | 2347.56 |
| Market-1 | burger | 2708.12 |
| Market-1 | chinese | 4040.50 |
| Market-1 | italian | 2809.86 |
| Market-1 | mediterranean | 4974.78 |
| Market-1 | pizza | 17510.74 |
| Market-1 | sandwich | 503.20 |
| Market-1 | thai | 2189.42 |
| Market-2 | alcohol | 2701.06 |
| Market-2 | american | 38526.11 |
| Market-2 | asian | 3684.62 |
| Market-2 | barbecue | 11190.24 |
| Market-2 | breakfast | 5042.25 |
| Market-2 | burger | 24792.11 |
| Market-2 | cafe | 1211.86 |
| Market-2 | chinese | 6438.59 |
| Market-2 | dessert | 28666.91 |
| Market-2 | greek | 17339.85 |
| Market-2 | hawaiian | 2943.23 |
| Market-2 | indian | 8044.14 |
| Market-2 | italian | 14863.39 |
| Market-2 | japanese | 2684.78 |
| Market-2 | korean | 3347.83 |
| Market-2 | mediterranean | 1751.94 |
| Market-2 | mexican | 45080.53 |
| Market-2 | middle-eastern | 2408.36 |
| Market-2 | other | 4093.00 |
| Market-2 | pasta | 3099.70 |
| Market-2 | pizza | 27505.56 |
| Market-2 | salad | 1503.97 |
| Market-2 | sandwich | 14081.46 |
| Market-2 | seafood | 6176.94 |
| Market-2 | steak | 7084.72 |
| Market-2 | sushi | 13513.00 |
| Market-2 | thai | 10923.95 |
| Market-2 | vietnamese | 5894.65 |
| Market-3 | pizza | 4092.86 |
| Market-3 | salad | 3131.82 |
| Market-4 | NA | 2174.66 |
| Market-4 | alcohol | 2640.33 |
| Market-4 | american | 7251.35 |
| Market-4 | asian | 3112.28 |
| Market-4 | bubble-tea | 2499.25 |
| Market-4 | burger | 19227.91 |
| Market-4 | catering | 7768.99 |
| Market-4 | chinese | 18118.21 |
| Market-4 | dessert | 23539.12 |
| Market-4 | dim-sum | 7483.75 |
| Market-4 | greek | 9621.21 |
| Market-4 | hawaiian | 1557.42 |
| Market-4 | indian | 15624.31 |
| Market-4 | italian | 4407.61 |
| Market-4 | japanese | 10809.51 |
| Market-4 | korean | 3626.33 |
| Market-4 | mediterranean | 16945.75 |
| Market-4 | mexican | 16682.12 |
| Market-4 | middle-eastern | 3459.90 |
| Market-4 | nepalese | 3458.57 |
| Market-4 | other | 12832.95 |
| Market-4 | pasta | 4906.74 |
| Market-4 | pizza | 22827.98 |
| Market-4 | salad | 17179.53 |
| Market-4 | sandwich | 2637.15 |
| Market-4 | seafood | 8919.89 |
| Market-4 | sushi | 7907.05 |
| Market-4 | thai | 17878.81 |
| Market-4 | vietnamese | 15467.80 |
| Market-5 | chinese | 4020.23 |
| Market-5 | indian | 4116.23 |
| Market-5 | vietnamese | 1109.76 |
| Market-6 | american | 2228.38 |
| Market-6 | dessert | 1660.37 |
| Market-6 | greek | 1392.34 |
| Market-6 | mexican | 2893.97 |
| Market-6 | other | 1185.64 |
| Market-6 | pizza | 3061.78 |
| Market-6 | sandwich | 2139.33 |

Lignes Silver relues : 50000
Stratégie : BroadcastHashJoin (explicit broadcast of 200 restaurants)


## Plan de jointure
Les sections Parsed/Analyzed décrivent le plan logique ; Optimized le travail de Catalyst ; Physical la stratégie exécutée. Repérer `BroadcastHashJoin` et `BroadcastExchange`.

In [6]:
print((ROOT / 'results/part3_explain_join.txt').read_text())

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [restaurant_id])
:- Relation [order_id#0,restaurant_id#1,customer_id#2,city#3,status#4,amount_usd#5,items_count#6,distance_km#7,courier_id#8,event_time#9,tenant_id#10,source_event_time#11,source_row#12,source_dataset#13,source_subtotal_cents#14,simulated_fields#15,run_id#16,scenario#17] json
+- ResolvedHint (strategy=broadcast)
   +- Project [restaurant_id#36, name#37 AS restaurant_name#124, restaurant_city#117, category#39, latitude#40, longitude#41]
      +- Project [restaurant_id#36, name#37, city#38 AS restaurant_city#117, category#39, latitude#40, longitude#41]
         +- Relation [restaurant_id#36,name#37,city#38,category#39,latitude#40,longitude#41] json

== Analyzed Logical Plan ==
restaurant_id: string, order_id: string, customer_id: string, city: string, status: string, amount_usd: decimal(12,2), items_count: int, distance_km: double, courier_id: string, event_time: timestamp, tenant_id: string, source_event_time: string, sour

## Validation indépendante
Un calcul Python standard vérifie les montants en centimes, les paniers moyens, le top 5 et les indicateurs par catégorie. Les journaux Spark fournissent les vrais jobs et stages. Les liens History Server restent consultables après la fin des applications.

In [7]:
run_script('validate_results.py')
run_script('collect_evidence.py')
display(read_result('validation.json'))
metrics = read_result('execution_metrics.json')
table([{'partie': part, 'jobs': value['number_of_jobs'], 'stages terminés': value['number_of_completed_stage_attempts'], 'interface': value['history_url']} for part, value in metrics.items()])

validate_results.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)
collect_evidence.py : resultats enregistres (activer RUN_PIPELINE pour recalculer)


{'status': 'PASS',
 'oracle': 'Python standard library, independent of Spark',
 'checks': ['50000 events and 200 restaurants',
  '1000 exact order/status duplicate rows',
  'Referential integrity and city consistency',
  'RDD = DataFrame = SQL = independent revenue totals',
  'Counts and average baskets',
  'Top 5 restaurants with deterministic tie breaking',
  'Category/city revenue',
  'Distance band counts',
  'Parquet read-back count and revenue (Spark assertions)'],
 'delivered_events': 25004,
 'revenue_usd': 688952.3}

| partie | jobs | stages terminés | interface |
| --- | --- | --- | --- |
| part1 | 1 | 1 | http://localhost:18080/history/app-20261114105747-0000/jobs/ |
| part2 | 5 | 8 | http://localhost:18080/history/app-20261114105804-0001/jobs/ |
| part3 | 75 | 79 | http://localhost:18080/history/app-20261114105828-0002/jobs/ |

## Exercice de préparation à la soutenance
Pourquoi `orders.withColumn('band', udf(...)).count()` ne suffit-il pas à mesurer le coût de l'UDF ? Quelle différence entre le nombre de Workers et le nombre d'Executors ?

**Réponse guidée :** Catalyst peut supprimer la colonne inutilisée ; il faut consommer sa valeur (ici groupBy). Un Worker est un service du cluster ; un Executor est un processus propre à une application, lancé sur un Worker.

**Piège courant :** les événements d'une même commande comportent plusieurs statuts. Sommer tous les statuts gonflerait le chiffre d'affaires. Ici le filtre DELIVERED est commun aux trois API, mais les doublons sont encore conservés.

**Extension :** comparer ensuite les résultats après dédoublonnage `(order_id, status)` et quantifier l'écart, sans remplacer les résultats exigés des parties 2/3.